## 授權與來源（License & Attribution）

本課程自編與改編部分：Copyright 2026 leonjyentub；授權：Apache License 2.0（`../LICENSE`）。
上游：Aurélien Géron, `ageron/handson-mlp`（Apache-2.0），commit `47eba45aacc85feae51ba7db68dd1ca66cb25e0a`。
另一來源：`leonjyentub/MachineLearning2025`（課程作者自有程式；其程式另採 Apache-2.0）。
本檔修改聲明：取材與參考 handson-mlp Ch.5／Appendix C；整合 MachineLearning2025 決策樹、hinge loss 與 SVM kernel。 教學版本重新編排與精簡實驗、補充課堂說明及輸出，並非未修改的原始上游檔案。
完整來源對照與第三方授權：`../SOURCES.md`、`../THIRD_PARTY_NOTICES.md`。
上述 Apache-2.0 不延伸至未取得授權的資料、圖片或其他第三方素材；保留所有適用的原始署名。

# 06｜決策樹與SVM補充

對應 `slides/06_決策樹與SVM補充.md`。本檔由 `11_決策樹與SVM.ipynb` 改名，程式與已執行輸出保留。

**Colab 示範**：直接上傳此 `.ipynb`，從上到下執行；第一個程式格會安裝缺少的套件，所需資料在使用時自動下載。NumPy 與 scikit-learn 實驗使用 CPU；這些程式沒有可直接切換的 CUDA 後端。輸出存於目前工作目錄。


In [ ]:
# 獨立執行：Colab 上傳此檔後，從這一格依序執行；無需掛載 Git 專案。
import importlib.util, subprocess, sys
_needed = {"numpy": "numpy>=1.26,<3", "pandas": "pandas>=2.0,<3", "matplotlib": "matplotlib>=3.7,<4", "scipy": "scipy>=1.11,<2", "sklearn": "scikit-learn>=1.4,<2", "joblib": "joblib>=1.3,<2", "cloudpickle": "cloudpickle>=3,<4", "ipywidgets": "ipywidgets>=8,<9"}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

from pathlib import Path
import hashlib, io, json, os, tarfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
# 本機在 programs/ 執行時沿用 data/、outputs/；Colab 則使用自己的工作目錄。
DATA = ROOT / ("data" if (ROOT / "pyproject.toml").exists() else "mlcourse_data")
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
SEED = 42
FAST = os.environ.get("MLCOURSE_PROFILE", "fast") != "full"
_SOURCE_COMMIT = "9e29abbbea6a3ff7250dea64c43c1d6c723095d7"
_SOURCES = {
    "housing.csv": (f"https://raw.githubusercontent.com/ageron/data/{_SOURCE_COMMIT}/housing.tgz", "2364609dc48bec7df3ba9dbb7041478e704ecddcee70ef1827ec3fc49d22c0cc"),
    "lifesat.csv": (f"https://raw.githubusercontent.com/ageron/data/{_SOURCE_COMMIT}/lifesat/lifesat.csv", "e247f7f6c19b0d8c55fb2af293f4fb3b710e31476534ad8e9b7e06ea546a2099"),
}

def setup():
    plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                         "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})
    OUT.mkdir(parents=True, exist_ok=True)
    return {"profile": "fast" if FAST else "full", "seed": SEED, "data": str(DATA)}

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    plt.gcf().savefig(folder / f"{name}.svg", bbox_inches="tight")
    plt.show()

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")

def data_path(name):
    if name not in (*_SOURCES, "mnist_784_v1.npz"):
        raise ValueError(f"未知資料檔：{name}")
    DATA.mkdir(parents=True, exist_ok=True)
    path = DATA / name
    if name in _SOURCES:
        url, expected_sha = _SOURCES[name]
        if not path.exists():
            print(f"下載 {name} …")
            with urllib.request.urlopen(url, timeout=180) as response:
                blob = response.read()
            if name == "housing.csv":
                with tarfile.open(fileobj=io.BytesIO(blob), mode="r:gz") as archive:
                    blob = archive.extractfile("housing/housing.csv").read()
            if hashlib.sha256(blob).hexdigest() != expected_sha:
                raise ValueError(f"資料校驗失敗：{name}")
            path.write_bytes(blob)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected_sha:
            raise ValueError(f"本機資料校驗失敗：{name}")
        info = {"url": url, "sha256": expected_sha}
    else:
        if not path.exists():
            print("從 OpenML 下載 MNIST（首次約需數分鐘）…")
            from sklearn.datasets import fetch_openml
            ds = fetch_openml("mnist_784", version=1, as_frame=False,
                              data_home=DATA / "openml_cache", parser="auto")
            X, y = ds.data.astype(np.uint8), ds.target.astype(np.uint8)
            assert X.shape == (70000, 784) and y.shape == (70000,)
            np.savez_compressed(path, X=X, y=y)
        with np.load(path) as archive:
            assert archive["X"].shape == (70000, 784)
            assert archive["y"].shape == (70000,)
        info = {"url": "https://www.openml.org/d/554",
                "sha256": hashlib.sha256(path.read_bytes()).hexdigest()}
    manifest_file = DATA / "manifest.json"
    manifest = json.loads(manifest_file.read_text()) if manifest_file.exists() else {}
    manifest[name] = info
    manifest_file.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n")
    return path

def housing_split():
    from sklearn.model_selection import train_test_split
    frame = pd.read_csv(data_path("housing.csv"))
    income = pd.cut(frame.median_income, [0, 1.5, 3, 4.5, 6, np.inf], labels=[1, 2, 3, 4, 5])
    train, test = train_test_split(frame, test_size=0.2, random_state=SEED, stratify=income)
    assert not set(train.index) & set(test.index)
    return train, test

setup()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## 1. 一棵可閱讀的 Iris 決策樹

`export_text` 印出從根到葉的規則；葉節點的 `predict_proba` 來自到達該葉的訓練樣本比例。

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
iris = load_iris()
Xtr, Xte, ytr, yte = train_test_split(iris.data, iris.target, test_size=.3, stratify=iris.target, random_state=SEED)
tree = DecisionTreeClassifier(max_depth=2, random_state=SEED).fit(Xtr, ytr)
print(export_text(tree, feature_names=list(iris.feature_names)))
print('leaf probabilities for first 2 test flowers:')
print(np.round(tree.predict_proba(Xte[:2]), 3))
fig, ax = plt.subplots(figsize=(9, 5))
plot_tree(tree, feature_names=iris.feature_names, class_names=list(iris.target_names), filled=True, ax=ax, fontsize=8)
savefig('11_iris_tree')

## 2. Gini 與 Entropy，以及投影片手算驗算

$\text{Gini}=1-\sum_k p_k^2$；$\text{Entropy}=-\sum_k p_k\log_2 p_k$。對應投影片活動「哪個節點比較純？」節點標籤數 `[4,0]`、`[2,2]`、`[3,1]`。

In [ ]:
def gini(counts):
    p = np.asarray(counts, float); p = p / p.sum()
    return float(1 - np.sum(p ** 2))
def entropy(counts):
    p = np.asarray(counts, float); p = p / p.sum()
    p = p[p > 0]
    return float(-np.sum(p * np.log2(p)))

nodes = {'[4,0]': [4, 0], '[2,2]': [2, 2], '[3,1]': [3, 1]}
tbl = pd.DataFrame({'Gini': {k: gini(v) for k, v in nodes.items()},
                    'Entropy': {k: entropy(v) for k, v in nodes.items()}})
assert np.allclose(tbl.Gini, [0, .5, .375])
assert np.allclose(tbl.Entropy, [0, 1, 0.81127812])
tbl

## 3. 手算一次分割：不能只比較左右平均

對應投影片活動「算一次分割」。父節點 `[4,4]`（8 筆），方案 A：左 `[3,0]`、右 `[1,4]`；方案 B：左 `[2,2]`、右 `[2,2]`。權重是樣本數比例。

In [ ]:
def weighted_gini(left, right):
    nl, nr = sum(left), sum(right); n = nl + nr
    return nl / n * gini(left) + nr / n * gini(right)
parent = gini([4, 4])
A = weighted_gini([3, 0], [1, 4]); B = weighted_gini([2, 2], [2, 2])
assert np.isclose(parent, .5) and np.isclose(A, .2) and np.isclose(B, .5)
assert np.isclose(parent - A, .3) and np.isclose(parent - B, 0.0)
pd.Series({'parent_Gini': parent, 'planA_weighted_Gini': A, 'planA_drop': parent - A,
           'planB_weighted_Gini': B, 'planB_drop': parent - B})

## 4. 正則化：彎月資料上的邊界

不加限制的樹會把訓練點框得很細。`max_depth`、`min_samples_leaf` 等把有效複雜度壓低（教材圖 5-3）。

In [ ]:
from sklearn.datasets import make_moons
Xm, ym = make_moons(n_samples=400, noise=.3, random_state=SEED)
Xmtr, Xmte, ymtr, ymte = train_test_split(Xm, ym, test_size=.3, random_state=SEED)
xx, yy = np.meshgrid(np.linspace(Xm[:, 0].min() - .5, Xm[:, 0].max() + .5, 300),
                     np.linspace(Xm[:, 1].min() - .5, Xm[:, 1].max() + .5, 300))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
rows = []
for ax, kw, name in [(axes[0], {}, 'no limit'), (axes[1], {'min_samples_leaf': 20}, 'min_samples_leaf=20')]:
    t = DecisionTreeClassifier(random_state=SEED, **kw).fit(Xmtr, ymtr)
    zz = t.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=.3, cmap='coolwarm')
    ax.scatter(Xmtr[:, 0], Xmtr[:, 1], c=ymtr, cmap='coolwarm', s=12, edgecolor='k', linewidth=.3)
    ax.set_title(f'{name}\ntrain={t.score(Xmtr, ymtr):.2f} test={t.score(Xmte, ymte):.2f}')
    rows.append({'model': name, 'depth': t.get_depth(), 'train_acc': t.score(Xmtr, ymtr), 'test_acc': t.score(Xmte, ymte)})
savefig('11_tree_regularization')
moons_reg = pd.DataFrame(rows).set_index('model')
assert moons_reg.loc['no limit', 'train_acc'] > moons_reg.loc['min_samples_leaf=20', 'train_acc']
moons_reg

## 5. 迴歸樹：葉節點輸出平均，因為平方損失

對應投影片活動「為什麼預測平均值？」葉內目標 `[1,2,6]`。

In [ ]:
leaf = np.array([1., 2., 6.])
mse_of = {c: float(np.mean((leaf - c) ** 2)) for c in [2, 3, 4]}
mae_of = {c: float(np.mean(np.abs(leaf - c))) for c in [2, 3, 4]}
assert np.isclose(mse_of[3], 14 / 3) and mse_of[3] == min(mse_of.values())   # 平均使 MSE 最小
assert mae_of[2] == min(mae_of.values())                                     # 中位數使 MAE 最小
print('MSE by constant:', {k: round(v, 3) for k, v in mse_of.items()}, '-> best = mean =', leaf.mean())
print('MAE by constant:', {k: round(v, 3) for k, v in mae_of.items()}, '-> best = median =', np.median(leaf))

from sklearn.tree import DecisionTreeRegressor
rng = np.random.default_rng(SEED)
xr = np.sort(rng.uniform(0, 1, 200))[:, None]
yr = np.sin(2 * np.pi * xr[:, 0]) + rng.normal(0, .2, 200)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, d in zip(axes, [2, 5]):
    rt = DecisionTreeRegressor(max_depth=d, random_state=SEED).fit(xr, yr)
    g = np.linspace(0, 1, 500)[:, None]
    ax.scatter(xr, yr, s=8, alpha=.4); ax.plot(g, rt.predict(g), 'r-')
    ax.set_title(f'regression tree, max_depth={d}')
savefig('11_regression_tree')

## 6. 座標方向與不穩定性

樹只能做垂直於座標軸的切割。旋轉特徵（例如用 PCA）後，同一份資料可能用更少的分割就分開（教材圖 5-7～5-9）。

In [ ]:
from sklearn.decomposition import PCA
rng = np.random.default_rng(SEED)
base = rng.uniform(-1, 1, (300, 2))
angle = np.deg2rad(30); R = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
Xrot = base @ R.T
lab = (base[:, 0] > 0).astype(int)            # 真實邊界在旋轉前的座標軸上
depth_raw = DecisionTreeClassifier(random_state=SEED).fit(Xrot, lab).get_depth()
Xpca = PCA(n_components=2, random_state=SEED).fit_transform(Xrot)
depth_pca = DecisionTreeClassifier(random_state=SEED).fit(Xpca, lab).get_depth()
# 小幅改動訓練資料 -> 規則可能整組改變
d1 = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(Xrot, lab)
d2 = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(Xrot[:-8], lab[:-8])
assert depth_pca <= depth_raw
print('tree depth on rotated axes:', depth_raw, ' | after PCA rotation:', depth_pca)
print('root feature changed after dropping 8 rows:', d1.tree_.feature[0] != d2.tree_.feature[0]
      or not np.isclose(d1.tree_.threshold[0], d2.tree_.threshold[0]))

## 7. 補充 SVM：軟間隔 C 與 RBF gamma

SVM 用「間隔」比較分隔面。軟間隔 `C` 越大越不容忍違規；RBF `gamma` 越大，每個支持向量影響範圍越小、邊界越彎（教材附錄 C）。

In [ ]:
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (gamma, C) in zip(axes, [(0.5, 1), (5, 1), (5, 100)]):
    svm = make_pipeline(StandardScaler(), SVC(kernel='rbf', gamma=gamma, C=C)).fit(Xmtr, ymtr)
    zz = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=.3, cmap='coolwarm')
    ax.scatter(Xmtr[:, 0], Xmtr[:, 1], c=ymtr, cmap='coolwarm', s=12, edgecolor='k', linewidth=.3)
    ax.set_title(f'RBF gamma={gamma}, C={C}\ntest acc={svm.score(Xmte, ymte):.2f}')
savefig('11_svm_rbf')

## 8. 同一切分，三種模型的 CV 比較

樹深度 `[2,4,8,None]`、標準化 logistic、標準化 RBF SVM，用同一份 moons 訓練資料做 5 折 CV。

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
rows = []
for name, est in {'tree_d2': DecisionTreeClassifier(max_depth=2, random_state=SEED),
                  'tree_d4': DecisionTreeClassifier(max_depth=4, random_state=SEED),
                  'tree_d8': DecisionTreeClassifier(max_depth=8, random_state=SEED),
                  'tree_full': DecisionTreeClassifier(random_state=SEED),
                  'logistic_std': make_pipeline(StandardScaler(), LogisticRegression()),
                  'svm_rbf_std': make_pipeline(StandardScaler(), SVC(kernel='rbf', C=1))}.items():
    cv = cross_val_score(est, Xmtr, ymtr, cv=5)
    est.fit(Xmtr, ymtr)
    rows.append({'model': name, 'CV_accuracy': cv.mean(), 'CV_sd': cv.std(ddof=1), 'test_accuracy': est.score(Xmte, ymte)})
compare = pd.DataFrame(rows).set_index('model')
report('trees_svm', {'gini_entropy_check': tbl.to_dict(), 'split_gain': {'A_drop': .3, 'B_drop': 0.0},
       'moons_regularization': moons_reg.to_dict(), 'tree_depth_raw_vs_pca': [int(depth_raw), int(depth_pca)],
       'model_comparison': compare.to_dict()})
compare

## 練習與參考答案

- 「純度」等於「預測正確」嗎？**否，純度只描述該節點的訓練標籤組成，不保證未見樣本正確。**
- 為什麼分割要看加權不純度而非左右平均？**兩子節點樣本數常不同；權重是樣本數比例。**
- 迴歸葉節點若改用絕對誤差，最佳常數是什麼？**中位數。**
- 小幅改動資料就換一套規則，說明什麼？**單棵樹變異大；集成（下一本）用來降低這種變異。**